In [ ]:
from transformers import pipeline as hf_pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import re
from typing import Optional
import torch

In [15]:
def load_model(model_name: str = "microsoft/Phi-3-mini-4k-instruct"):
    """
    Load a HuggingFace text-generation pipeline.

    We use Phi-3-mini because:
    - Small (3.8B params), runs on CPU or consumer GPU
    - Instruction-tuned (understands chat format)
    - Good reasoning for its size

    Alternatives if you have more VRAM:
    - "mistralai/Mistral-7B-Instruct-v0.2" (7B)
    - "meta-llama/Meta-Llama-3-8B-Instruct" (8B, requires HF login)
    """

    print(f"Loading model: {model_name}")
    print("This may take a few minutes on first run (downloading ~2GB)...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    pipe = hf_pipeline(
        "text-generation",
        model=model_name,
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        device_map="auto",  # auto-selects CPU/GPU
        max_new_tokens=256,
        do_sample=False,    # deterministic for tutorials
        temperature=1.0,    # irrelevant when do_sample=False
    )
    print("Model loaded successfully!\n")
    return pipe


def generate(pipe, messages: list[dict], max_new_tokens: int = 256) -> str:
    """
    Generate a response from a list of chat messages.

    Messages format:
        [{"role": "system", "content": "..."},
         {"role": "user",   "content": "..."}]

    Returns the assistant's response as a string.
    """
    output = pipe(messages, max_new_tokens=max_new_tokens)
    # Extract the generated text (remove the prompt from the output)
    response = output[0]["generated_text"]
    # The last message in the conversation is the assistant's response
    if isinstance(response, list):
        return response[-1]["content"]
    return response



## Zero shot prompting

In [16]:
def demo_zero_shot(pipe):
    """
    Zero-shot prompting: ask directly without any examples.
    The model relies entirely on knowledge from pretraining.

    Think of this as maximum entropy inference — we provide minimal
    prior information and let the model's learned distribution do the work.
    """
    print("=" * 60)
    print("SECTION 1: Zero-Shot Prompting")
    print("=" * 60)

    # 1a. Basic zero-shot — simple factual question
    print("\n--- 1a. Basic Zero-Shot ---")
    messages = [
        {"role": "user", "content": "What is the band gap of silicon at room temperature?"}
    ]
    response = generate(pipe, messages)
    print(f"Prompt: {messages[0]['content']}")
    print(f"Response: {response}\n")

    # 1b. Zero-shot with role assignment (persona injection)
    print("--- 1b. Zero-Shot with Role Assignment ---")
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert materials scientist specializing in semiconductors. "
                "Be precise, cite numbers, and use SI units."
            )
        },
        {
            "role": "user",
            "content": "What is the band gap of silicon at room temperature?"
        }
    ]
    response = generate(pipe, messages)
    print(f"System: {messages[0]['content'][:60]}...")
    print(f"Prompt: {messages[1]['content']}")
    print(f"Response: {response}\n")

    # Key insight: the system message acts like a prior — it biases the
    # model's output distribution toward more expert-like responses.



## Few shot prompting

In [ ]:
def demo_few_shot(pipe):
    """
    Few-shot prompting: provide examples (demonstrations) before your query.

    This is in-context learning — the model updates its implicit distribution
    based on the examples in the prompt, without gradient updates.

    Analogy to your ML background: think of examples as pseudo-observations
    that shift the model's posterior over response formats.
    """
    print("=" * 60)
    print("SECTION 2: Few-Shot Prompting")
    print("=" * 60)

    # 2a. Few-shot for classification
    print("\n--- 2a. Few-Shot Classification ---")
    messages = [
        {
            "role": "system",
            "content": "Classify materials by their primary bonding type."
        },
        {
            "role": "user",
            "content": (
                "Examples:\n"
                "Material: NaCl -> Bonding: Ionic\n"
                "Material: Diamond -> Bonding: Covalent\n"
                "Material: Copper -> Bonding: Metallic\n"
                "Material: Water -> Bonding: Covalent\n\n"
                "Now classify:\n"
                "Material: Silicon -> Bonding:"
            )
        }
    ]
    response = generate(pipe, messages, max_new_tokens=50)
    print(f"Response: {response}")

    # 2b. Few-shot for structured extraction
    print("\n--- 2b. Few-Shot Structured Extraction ---")
    messages = [
        {
            "role": "system",
            "content": "Extract material properties from text. Always respond in the exact format shown."
        },
        {
            "role": "user",
            "content": (
                "Examples:\n"
                'Text: "Gold melts at 1064°C and has density 19.3 g/cm³"\n'
                'Output: {"material": "Gold", "melting_point_C": 1064, "density_g_cm3": 19.3}\n\n'
                'Text: "Copper has electrical conductivity of 5.96e7 S/m and melts at 1085°C"\n'
                'Output: {"material": "Copper", "melting_point_C": 1085, "conductivity_S_m": 5.96e7}\n\n'
                'Now extract from:\n'
                'Text: "Aluminum has density 2.7 g/cm³ and melting point 660°C"\n'
                'Output:'
            )
        }
    ]
    response = generate(pipe, messages, max_new_tokens=100)
    print(f"Response: {response}\n")



## Chain of thought prompting

In [17]:

def demo_chain_of_thought(pipe):
    """
    Chain-of-Thought (CoT): elicit step-by-step reasoning before the final answer.

    Why it works: Forces the model to allocate more compute (tokens) to
    intermediate reasoning steps, similar to iterative refinement in optimization.

    Particularly valuable for:
    - Multi-step calculations
    - Logical deduction
    - Scientific problem-solving
    """
    print("=" * 60)
    print("SECTION 3: Chain-of-Thought Prompting")
    print("=" * 60)

    # 3a. Direct answer (baseline)
    print("\n--- 3a. Direct Answer (Baseline) ---")
    messages = [
        {
            "role": "user",
            "content": (
                "A steel beam is at 20°C. The thermal expansion coefficient of steel "
                "is 12e-6 /°C. If the beam is 10 meters long and heated to 200°C, "
                "what is the new length?"
            )
        }
    ]
    response_direct = generate(pipe, messages, max_new_tokens=100)
    print(f"Direct response: {response_direct}")

    # 3b. Chain-of-Thought (improved)
    print("\n--- 3b. Chain-of-Thought ---")
    messages = [
        {
            "role": "system",
            "content": "Solve physics problems step by step. Show your work clearly."
        },
        {
            "role": "user",
            "content": (
                "A steel beam is at 20°C. The thermal expansion coefficient of steel "
                "is 12e-6 /°C. If the beam is 10 meters long and heated to 200°C, "
                "what is the new length?\n\n"
                "Let's think step by step:\n"
                "Step 1: Identify the formula for thermal expansion.\n"
                "Step 2: Identify given values.\n"
                "Step 3: Calculate the change in length.\n"
                "Step 4: Calculate the new length.\n"
                "Step 5: State the answer with units."
            )
        }
    ]
    response_cot = generate(pipe, messages, max_new_tokens=300)
    print(f"CoT response: {response_cot}\n")

    # 3c. Zero-Shot CoT (just add "think step by step")
    print("--- 3c. Zero-Shot CoT (Magic Phrase) ---")
    messages = [
        {
            "role": "user",
            "content": (
                "If we mix 60% iron, 20% chromium, and 20% nickel by weight, "
                "approximately what type of alloy do we get and what are its key properties? "
                "Think step by step."
            )
        }
    ]
    response_zs_cot = generate(pipe, messages, max_new_tokens=200)
    print(f"Zero-shot CoT response: {response_zs_cot}\n")



## Prompt Template

In [18]:

class PromptTemplate:
    """
    A reusable, parameterized prompt template.

    Similar to a function with default arguments — lets you standardize
    prompt structure while varying the inputs.
    """

    def __init__(self, template: str, input_variables: list[str]):
        self.template = template
        self.input_variables = input_variables

    def format(self, **kwargs) -> str:
        """Fill in the template with provided values."""
        # Validate that all required variables are provided
        missing = set(self.input_variables) - set(kwargs.keys())
        if missing:
            raise ValueError(f"Missing template variables: {missing}")
        return self.template.format(**kwargs)


def demo_prompt_templates(pipe):
    """
    Prompt templates standardize your prompts for reproducibility
    and batch processing — essential for building pipelines.
    """
    print("=" * 60)
    print("SECTION 4: Prompt Templates")
    print("=" * 60)

    # Define reusable templates
    material_property_template = PromptTemplate(
        template=(
            "You are a materials science database. Provide accurate, concise information.\n\n"
            "Material: {material}\n"
            "Property requested: {property}\n"
            "Units: {units}\n"
            "Answer (number and brief context only):"
        ),
        input_variables=["material", "property", "units"]
    )

    comparison_template = PromptTemplate(
        template=(
            "Compare {material_a} and {material_b} in terms of {criterion}. "
            "Format your response as:\n"
            "{material_a}: [value/description]\n"
            "{material_b}: [value/description]\n"
            "Verdict: [which is better for {use_case} and why in one sentence]"
        ),
        input_variables=["material_a", "material_b", "criterion", "use_case"]
    )

    # Use the property template with different materials
    queries = [
        {"material": "Silicon", "property": "band gap", "units": "eV"},
        {"material": "GaAs",    "property": "band gap", "units": "eV"},
        {"material": "Copper",  "property": "electrical resistivity", "units": "Ω·m"},
    ]

    print("\n--- Material Property Queries ---")
    for q in queries:
        prompt_text = material_property_template.format(**q)
        messages = [{"role": "user", "content": prompt_text}]
        response = generate(pipe, messages, max_new_tokens=80)
        print(f"{q['material']} - {q['property']}: {response.strip()[:100]}")

    # Use the comparison template
    print("\n--- Material Comparison ---")
    prompt_text = comparison_template.format(
        material_a="Steel",
        material_b="Aluminum",
        criterion="strength-to-weight ratio",
        use_case="aerospace structural components"
    )
    messages = [{"role": "user", "content": prompt_text}]
    response = generate(pipe, messages, max_new_tokens=150)
    print(f"Comparison result:\n{response}\n")


In [ ]:

def demo_structured_output(pipe):
    """
    Force the model to generate structured outputs (JSON, YAML, etc.)
    by including format instructions and examples in the prompt.

    Critical for downstream programmatic processing.
    """
    print("=" * 60)
    print("SECTION 5: Structured Output Generation")
    print("=" * 60)

    # 5a. JSON output with schema specification
    print("\n--- 5a. JSON Output ---")
    json_schema = {
        "material_name": "string",
        "crystal_structure": "string (e.g., FCC, BCC, HCP)",
        "melting_point_K": "number",
        "applications": ["string"],
        "is_semiconductor": "boolean"
    }

    messages = [
        {
            "role": "system",
            "content": (
                "You are a materials database API. Always respond with valid JSON only. "
                "No explanation, no markdown, just the JSON object."
            )
        },
        {
            "role": "user",
            "content": (
                f"Return a JSON object for Titanium with this exact schema:\n"
                f"{json.dumps(json_schema, indent=2)}\n\n"
                "Fill all fields with accurate values. Applications list should have 3 items."
            )
        }
    ]
    response = generate(pipe, messages, max_new_tokens=200)
    print(f"Raw response: {response}")

    # Try to parse the JSON
    try:
        # Extract JSON from response (handle cases where model adds extra text)
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            print(f"Parsed successfully: {json.dumps(parsed, indent=2)}")
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        print("Tip: Try adding 'Respond with ONLY the JSON, no other text' to the prompt")

    # 5b. Structured list output
    print("\n--- 5b. Structured List Output ---")
    messages = [
        {
            "role": "user",
            "content": (
                "List the top 5 metals by electrical conductivity. "
                "Format as a numbered list:\n"
                "1. [Metal] - [Conductivity in MS/m] - [Main use]\n"
                "2. ...\n\n"
                "Start your response with '1.' only."
            )
        }
    ]
    response = generate(pipe, messages, max_new_tokens=150)
    print(f"Response:\n{response}\n")


# ─────────────────────────────────────────────────────────────
# MAIN: RUN ALL DEMOS
# ─────────────────────────────────────────────────────────────

def main():
    print("\n" + "=" * 60)
    print("DAY 1: Foundations of Prompting")
    print("IBM AI Periodic Table — Element: Pr")
    print("=" * 60 + "\n")

    # Load the model once (reuse for all sections)
    pipe = load_model()

    # Run all demonstrations
    demo_zero_shot(pipe)
    demo_few_shot(pipe)
    demo_chain_of_thought(pipe)
    demo_prompt_templates(pipe)
    demo_structured_output(pipe)

    print("\n" + "=" * 60)
    print("Tutorial complete! Proceed to exercises.py")
    print("=" * 60)


if __name__ == "__main__":
    main()

### Making use if above techniques to extract useful info as materials engineer!

In [ ]:
def load_pipeline():
    """Load the model"""
    pipe = hf_pipeline(
        "text-generation",
        model="microsoft/Phi-3-mini-4k-instruct",
        torch_dtype=torch.float16,
        device_map="auto",
        max_new_tokens=256,
        do_sample=False,
    )
    return pipe


def generate(pipe, messages, max_new_tokens=256):
    output = pipe(messages, max_new_tokens=max_new_tokens)
    response = output[0]["generated_text"]
    if isinstance(response, list):
        return response[-1]["content"]
    return response


### Pr (Prompting) Exercise

To write a system prompt that makes the model respond as a "Senior Materials Scientist in semiconductor devices".

In [12]:
def exercise_1(pipe):

    print("\n--- EXERCISE 1: Role Prompting ---")

    system_prompt = """ Behave as a "Senior Materials Scientist with 20 years of experience" not a general assistant or 2nd class graduate student. Additionally,
    - Be quantitave and use technical terminology,
    - Always provide numerical values with units
    - Use the temperature when discussing temperature-dependent properties
    - Be concise (3 sentences max).
    """

    user_question = "How does doping affect silicon's electrical properties?"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_question}
    ]

    response = generate(pipe, messages, max_new_tokens=150)
    print(f"Response: {response}")



In [13]:
pipe = load_pipeline()
    
exercise_1(pipe)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- EXERCISE 1: Role Prompting ---
Response:  Doping silicon with impurities such as phosphorus (n-type) or boron (p-type) introduces additional charge carriers, significantly increasing its electrical conductivity. For instance, doping silicon with 1 part per million (ppm) of phosphorus can increase its conductivity by approximately 10^3 S/m. At room temperature (25°C), this doping can shift the Fermi level closer to the conduction band in n-type silicon, enhancing electron mobility and thus the material's overall conductivity.
